# A RAG pipeline from scratch

A coworking space wants a member bot that answers questions about room bookings, guests and mail from its
handbook. In this notebook you run the lesson's pipeline for real: score every handbook chunk against a question
with NumPy, keep the top 2, build the prompt, and replay the model's recorded answers. Then you rebuild the same
pipeline in LangChain and check that it sends the very same messages.

About 5 minutes. Run the cells from top to bottom on a free CPU runtime. No account and no API key are needed.
One cell near the end downloads a small embedding model (about 90 MB).

## 1. Install two libraries

Colab already has NumPy and PyTorch. The next cell adds the versions the lesson ran: langchain-core (for the
LangChain part) and sentence-transformers (to make vectors yourself near the end).

**Look for:** no error. pip may print a few lines about other packages; you can ignore them.

In [1]:
%pip install -q langchain-core==1.8.0 sentence-transformers==6.3.0

Note: you may need to restart the kernel to use updated packages.


## 2. The handbook and the questions

The same made-up handbook as the lesson: ten short paragraphs, called **chunks**, each with an id. Then the four
member questions, and the system instruction the pipeline puts first in every prompt.

**Look for:** 10 chunks and 4 questions.

In [2]:
HANDBOOK = [
    ('rooms-booking', "Meeting rooms are booked in the member app, up to 14 days ahead. A booking is held for 15 minutes after its start time. If nobody checks in on the tablet by the room door, the room is released for others."),
    ('rooms-hours', "Every member gets 6 meeting-room hours each month. Extra hours are charged to your monthly invoice. Unused hours do not carry over to the next month."),
    ('rooms-cancel', "Cancel a room booking at least 2 hours before it starts and the hours go back to your monthly allowance. Later cancellations and no-shows still count against your hours."),
    ('guests-daily', "Members may bring up to 2 guests per day. Register each guest in the app before they arrive, and the front desk prints a day pass. Guests must leave by 18:00 and cannot book rooms themselves."),
    ('guests-events', "For a meeting or event with more than 2 guests, book the event hall instead. Event hall requests need at least 7 days' notice and are approved by the community manager."),
    ('mail-arrival', "Letters and parcels addressed to members are kept at the front desk. You get an app notification when something arrives. Parcels heavier than 20 kg cannot be accepted."),
    ('mail-holding', "The front desk holds mail for 30 days. After 30 days, letters are returned to the sender and parcels are given back to the courier."),
    ('mail-forward', "Mail forwarding to another address is available on the Business plan only. Forwarded mail is sent once a week, on Mondays."),
    ('wifi', "The Wi-Fi password changes on the first day of each month. The current password is shown on the home screen of the member app."),
    ('access', "The building is open from 07:00 to 22:00 on weekdays. Members on a 24/7 plan can enter at any time with their key card."),
]

QUESTIONS = {
    'q-guests': "Can my two clients come in on Friday afternoon to meet me?",
    'q-noshow': "I missed my 10am room booking yesterday. Do I get those hours back?",
    'q-parcel': "A parcel came for me three weeks ago while I was travelling. Is it still at the desk?",
    'q-dog': "Can I bring my dog to work with me?",
}

SYSTEM = ("You answer questions from members of a coworking space. Use only the handbook excerpts "
          "in the user message. If they do not contain the answer, say that the handbook does not say, "
          "and suggest asking the front desk. Answer in at most three sentences.")

chunk_ids = [cid for cid, text in HANDBOOK]
chunk_texts = [text for cid, text in HANDBOOK]
print("chunks:", len(HANDBOOK))
print("questions:", len(QUESTIONS))
for cid in chunk_ids:
    print(" ", cid)

chunks: 10
questions: 4
  rooms-booking
  rooms-hours
  rooms-cancel
  guests-daily
  guests-events
  mail-arrival
  mail-holding
  mail-forward
  wifi
  access


## 3. The precomputed vectors

Before any question came in, the lesson's run turned every chunk and every question into a vector with a small
embedding model (all-minilm in Ollama, 384 numbers per text). Those are the **precomputed vectors**. Colab has no
Ollama, so the next cell carries the exact numbers from that run, packed into one long line of text. You do not
need to read the line. The vectors have length 1, so a dot product of two of them is their cosine similarity.

**Look for:** 10 chunk vectors and 4 question vectors, 384 numbers each. The length of the first chunk vector is
1.0, and the rooms-booking vector starts with 0.0104, -0.0513.

In [3]:
import base64, json
import numpy as np

np.random.seed(0)  # nothing in this notebook is random; the seed is here only as a habit

# 14 vectors from the lesson's run (Ollama all-minilm): the 10 chunks, then the 4 questions, as float32 bytes.
VECTORS = "..."

all_vecs = np.frombuffer(base64.b64decode(VECTORS), dtype="<f4").reshape(14, 384).astype(np.float64)
handbook_vecs = all_vecs[:10]                                   # 10 rows, 384 columns
question_vecs = dict(zip(QUESTIONS, all_vecs[10:]))

print(f"{len(handbook_vecs)} chunk vectors, {len(question_vecs)} question vectors, "
      f"{handbook_vecs.shape[1]} numbers each")
print("length of the first chunk vector:", round(float(np.linalg.norm(handbook_vecs[0])), 6))
print("first 5 numbers of the rooms-booking vector:", handbook_vecs[0][:5].round(4).tolist())

10 chunk vectors, 4 question vectors, 384 numbers each
length of the first chunk vector: 1.0
first 5 numbers of the rooms-booking vector: [0.0104, -0.0513, -0.0319, -0.0314, -0.0488]


## 4. Retrieve: score, sort, keep the top k

This is the **retrieve** step, line by line, for the parcel question. One matrix product gives one score per
chunk. `np.argsort(-scores)` gives the chunk positions from the highest score to the lowest, and `order[:k]`
keeps the first k. Then the same scores for all four questions, best first.

**Look for:** for the parcel question, `top` is `[6, 5]`: mail-holding (0.5442) and mail-arrival (0.5318), close
to each other. For the dog question the best score is only 0.1675, far lower than the other three questions.

In [4]:
k = 2
question_vec = question_vecs["q-parcel"]
scores = handbook_vecs @ question_vec
order = np.argsort(-scores)
top = order[:k]
print("scores:", scores.round(4).tolist())
print("order: ", order.tolist())
print("top:   ", top.tolist(), "=", [chunk_ids[i] for i in top])

print("\nEvery question, all chunks, best first:")
for qid in QUESTIONS:
    s = handbook_vecs @ question_vecs[qid]
    ranked = sorted(zip(chunk_ids, s.tolist()), key=lambda t: -t[1])
    print(qid, [(cid, round(v, 4)) for cid, v in ranked[:4]], "...")

scores: [0.1682, 0.0289, 0.1309, 0.082, 0.0659, 0.5318, 0.5442, 0.3181, 0.0803, 0.0401]
order:  [6, 5, 7, 0, 2, 3, 8, 4, 9, 1]
top:    [6, 5] = ['mail-holding', 'mail-arrival']

Every question, all chunks, best first:
q-guests [('guests-daily', 0.4325), ('rooms-booking', 0.3981), ('guests-events', 0.3513), ('access', 0.3283)] ...
q-noshow [('rooms-cancel', 0.66), ('rooms-booking', 0.523), ('guests-daily', 0.4644), ('access', 0.3977)] ...
q-parcel [('mail-holding', 0.5442), ('mail-arrival', 0.5318), ('mail-forward', 0.3181), ('rooms-booking', 0.1682)] ...
q-dog [('mail-arrival', 0.1675), ('mail-forward', 0.1671), ('guests-events', 0.1142), ('guests-daily', 0.1103)] ...


## 5. Assemble: the prompt

`build_prompt` is the function from the lesson's code screen, unchanged. It does retrieve, then **assemble**:
each chosen chunk goes into a `<chunk>` tag with its id, and the excerpts and the question go into the user
message. The system instruction comes first.

**Look for:** a system message, then a user message with the two parcel chunks (mail-holding first) and the
question at the end. This is the prompt you saw on the lesson's screen 3.

In [5]:
def build_prompt(member_question, k=2):
    question_vec = question_vecs[member_question["id"]]
    scores = handbook_vecs @ question_vec
    order = np.argsort(-scores)
    top = order[:k]
    excerpts = "\n".join(f'<chunk id="{chunk_ids[i]}">{chunk_texts[i]}</chunk>' for i in top)
    user = f"Handbook excerpts:\n{excerpts}\n\nQuestion: {member_question['text']}"
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]


def ask(qid, k=2):
    return build_prompt({"id": qid, "text": QUESTIONS[qid]}, k)


parcel_messages = ask("q-parcel")
for m in parcel_messages:
    print(f"--- {m['role']}")
    print(m["content"])

--- system
You answer questions from members of a coworking space. Use only the handbook excerpts in the user message. If they do not contain the answer, say that the handbook does not say, and suggest asking the front desk. Answer in at most three sentences.
--- user
Handbook excerpts:
<chunk id="mail-holding">The front desk holds mail for 30 days. After 30 days, letters are returned to the sender and parcels are given back to the courier.</chunk>
<chunk id="mail-arrival">Letters and parcels addressed to members are kept at the front desk. You get an app notification when something arrives. Parcels heavier than 20 kg cannot be accepted.</chunk>

Question: A parcel came for me three weeks ago while I was travelling. Is it still at the desk?


## 6. Answer: replay the recorded model answers

The third step sends the messages to a chat model. You have no API key, so the next cell holds the five answers
the lesson recorded from a small hosted model (temperature 0), each with the exact messages that were sent.
`ask_model` looks up the messages you pass in. It finds an answer only if your messages are **exactly** the ones
that were sent, character for character.

**Look for:** the parcel answer says yes, the desk holds mail for 30 days. Then the guest question twice: the
question alone gets a confident guess with no rules; with the handbook chunks it gets the two rules (register
in the app, leave by 6 PM).

In [6]:
RECORDINGS = [
 {
  "name": "bare-q-guests",
  "model": "google/gemini-2.5-flash-lite",
  "recorded": "2026-11-13",
  "temperature": 0,
  "messages": [
   {
    "role": "system",
    "content": "You answer questions from members of a coworking space. Answer in at most three sentences."
   },
   {
    "role": "user",
    "content": "Can my two clients come in on Friday afternoon to meet me?"
   }
  ],
  "answer": "Yes, absolutely! Please let us know the approximate number of people attending and if you'll need any specific meeting rooms or amenities. We'll do our best to accommodate your clients' visit."
 },
 {
  "name": "answer-q-guests",
  "model": "google/gemini-2.5-flash-lite",
  "recorded": "2026-11-13",
  "temperature": 0,
  "messages": [
   {
    "role": "system",
    "content": "You answer questions from members of a coworking space. Use only the handbook excerpts in the user message. If they do not contain the answer, say that the handbook does not say, and suggest asking the front desk. Answer in at most three sentences."
   },
   {
    "role": "user",
    "content": "Handbook excerpts:\n<chunk id=\"guests-daily\">Members may bring up to 2 guests per day. Register each guest in the app before they arrive, and the front desk prints a day pass. Guests must leave by 18:00 and cannot book rooms themselves.</chunk>\n<chunk id=\"rooms-booking\">Meeting rooms are booked in the member app, up to 14 days ahead. A booking is held for 15 minutes after its start time. If nobody checks in on the tablet by the room door, the room is released for others.</chunk>\n\nQuestion: Can my two clients come in on Friday afternoon to meet me?"
   }
  ],
  "answer": "Yes, your two clients can come in on Friday afternoon to meet you. They must be registered in the app before they arrive, and they need to leave by 6 PM."
 },
 {
  "name": "answer-q-noshow",
  "model": "google/gemini-2.5-flash-lite",
  "recorded": "2026-11-13",
  "temperature": 0,
  "messages": [
   {
    "role": "system",
    "content": "You answer questions from members of a coworking space. Use only the handbook excerpts in the user message. If they do not contain the answer, say that the handbook does not say, and suggest asking the front desk. Answer in at most three sentences."
   },
   {
    "role": "user",
    "content": "Handbook excerpts:\n<chunk id=\"rooms-cancel\">Cancel a room booking at least 2 hours before it starts and the hours go back to your monthly allowance. Later cancellations and no-shows still count against your hours.</chunk>\n<chunk id=\"rooms-booking\">Meeting rooms are booked in the member app, up to 14 days ahead. A booking is held for 15 minutes after its start time. If nobody checks in on the tablet by the room door, the room is released for others.</chunk>\n\nQuestion: I missed my 10am room booking yesterday. Do I get those hours back?"
   }
  ],
  "answer": "The handbook states that cancellations must be made at least 2 hours before the booking starts to get the hours back. Later cancellations and no-shows still count against your hours. The handbook does not specify exceptions for missed bookings."
 },
 {
  "name": "answer-q-parcel",
  "model": "google/gemini-2.5-flash-lite",
  "recorded": "2026-11-13",
  "temperature": 0,
  "messages": [
   {
    "role": "system",
    "content": "You answer questions from members of a coworking space. Use only the handbook excerpts in the user message. If they do not contain the answer, say that the handbook does not say, and suggest asking the front desk. Answer in at most three sentences."
   },
   {
    "role": "user",
    "content": "Handbook excerpts:\n<chunk id=\"mail-holding\">The front desk holds mail for 30 days. After 30 days, letters are returned to the sender and parcels are given back to the courier.</chunk>\n<chunk id=\"mail-arrival\">Letters and parcels addressed to members are kept at the front desk. You get an app notification when something arrives. Parcels heavier than 20 kg cannot be accepted.</chunk>\n\nQuestion: A parcel came for me three weeks ago while I was travelling. Is it still at the desk?"
   }
  ],
  "answer": "Yes, your parcel should still be at the front desk. The front desk holds mail for 30 days, and your parcel arrived three weeks ago."
 },
 {
  "name": "answer-q-dog",
  "model": "google/gemini-2.5-flash-lite",
  "recorded": "2026-11-13",
  "temperature": 0,
  "messages": [
   {
    "role": "system",
    "content": "You answer questions from members of a coworking space. Use only the handbook excerpts in the user message. If they do not contain the answer, say that the handbook does not say, and suggest asking the front desk. Answer in at most three sentences."
   },
   {
    "role": "user",
    "content": "Handbook excerpts:\n<chunk id=\"mail-arrival\">Letters and parcels addressed to members are kept at the front desk. You get an app notification when something arrives. Parcels heavier than 20 kg cannot be accepted.</chunk>\n<chunk id=\"mail-forward\">Mail forwarding to another address is available on the Business plan only. Forwarded mail is sent once a week, on Mondays.</chunk>\n\nQuestion: Can I bring my dog to work with me?"
   }
  ],
  "answer": "The handbook does not say. Please ask the front desk."
 }
]


def ask_model(messages):
    """Replays a recorded answer. Only the exact messages that were sent have one."""
    for rec in RECORDINGS:
        if rec["messages"] == messages:
            return rec["answer"], f'recorded {rec["recorded"]}, {rec["model"]}, temperature {rec["temperature"]}'
    return None, "no recording for these exact messages (they differ from what was sent)"


answer, note = ask_model(parcel_messages)
print("PARCEL:", answer, f"  [{note}]")

bare = [{"role": "system", "content": "You answer questions from members of a coworking space. Answer in at most three sentences."},
        {"role": "user", "content": QUESTIONS["q-guests"]}]
print("\nGUESTS, question alone:      ", ask_model(bare)[0])
print("GUESTS, with the handbook:   ", ask_model(ask("q-guests"))[0])

PARCEL: Yes, your parcel should still be at the front desk. The front desk holds mail for 30 days, and your parcel arrived three weeks ago.   [recorded 2026-11-13, google/gemini-2.5-flash-lite, temperature 0]

GUESTS, question alone:       Yes, absolutely! Please let us know the approximate number of people attending and if you'll need any specific meeting rooms or amenities. We'll do our best to accommodate your clients' visit.
GUESTS, with the handbook:    Yes, your two clients can come in on Friday afternoon to meet you. They must be registered in the app before they arrive, and they need to leave by 6 PM.


## 7. A question the handbook does not cover

The dog question goes through the same pipeline. A vector search has no idea of "nothing fits": it ranks every
chunk and keeps the top k.

**Look for:** 2 chunks still come back (mail-arrival and mail-forward, scores about 0.17). The recorded answer
says the handbook does not say, because the instruction gave the model a way out.

In [7]:
dog_messages = ask("q-dog")
s = handbook_vecs @ question_vecs["q-dog"]
for i in np.argsort(-s)[:2]:
    print(f"retrieved: {chunk_ids[i]:<14} score {s[i]:.4f}")
print("chunks in the prompt:", dog_messages[1]["content"].count("<chunk "))
print("answer:", ask_model(dog_messages)[0])

retrieved: mail-arrival   score 0.1675
retrieved: mail-forward   score 0.1671
chunks in the prompt: 2
answer: The handbook does not say. Please ask the front desk.


## 8. The same pipeline in LangChain

These are the lesson's LangChain lines. One change: the lesson used `OllamaEmbeddings(model="all-minilm")`, and
Colab has no Ollama. `StoredVectors` is a tiny LangChain embedding class that hands back the same precomputed
vectors from step 3, so LangChain sees exactly the numbers the lesson's run saw. Everything else is the same:
the in-memory store (cosine similarity), the retriever with `k` 2, the template and `join_chunks`.

**Look for:** the same top 2 chunks with the same scores as step 4, and **identical** for all four questions:
LangChain built the same messages as `build_prompt`, and the same as the recorded requests.

In [8]:
from langchain_core.embeddings import Embeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.vectorstores import InMemoryVectorStore


class StoredVectors(Embeddings):
    """Gives back the precomputed vector of each known text (instead of calling Ollama)."""
    def __init__(self):
        self.table = dict(zip(chunk_texts, handbook_vecs.tolist()))
        self.table.update({QUESTIONS[q]: v.tolist() for q, v in question_vecs.items()})
    def embed_documents(self, texts):
        return [self.table[t] for t in texts]
    def embed_query(self, text):
        return self.table[text]


store = InMemoryVectorStore.from_texts(chunk_texts, StoredVectors(),
                                       metadatas=[{"id": i} for i in chunk_ids])
retriever = store.as_retriever(search_kwargs={"k": 2})
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    ("human", "Handbook excerpts:\n{excerpts}\n\nQuestion: {question}")])
def join_chunks(docs):
    return "\n".join(f'<chunk id="{d.metadata["id"]}">{d.page_content}</chunk>' for d in docs)
up_to_prompt = {"excerpts": retriever | join_chunks, "question": RunnablePassthrough()} | prompt

role = {"system": "system", "human": "user"}
recorded = {r["name"]: r["messages"] for r in RECORDINGS}
for qid, text in QUESTIONS.items():
    hits = store.similarity_search_with_score(text, k=2)
    lc = [{"role": role[m.type], "content": m.content} for m in up_to_prompt.invoke(text).to_messages()]
    same = lc == ask(qid) == recorded["answer-" + qid]
    print(qid, [(d.metadata["id"], round(sc, 4)) for d, sc in hits], "identical" if same else "DIFFERENT")

q-guests [('guests-daily', 0.4325), ('rooms-booking', 0.3981)] identical
q-noshow [('rooms-cancel', 0.66), ('rooms-booking', 0.523)] identical
q-parcel [('mail-holding', 0.5442), ('mail-arrival', 0.5318)] identical
q-dog [('mail-arrival', 0.1675), ('mail-forward', 0.1671)] identical


## 9. The whole chain

`chain = up_to_prompt | llm | StrOutputParser()` is the lesson's last LangChain line. Here `llm` replays the
recorded answers through `ask_model` (in your own project it would be a real chat model object). After that, the
cell prints what the lesson's run got when it plugged in a very small local model (Ollama gemma3:270m) instead.
That model cannot run here, so its answers are shown as data from the run.

**Look for:** the hosted model's dog answer says the handbook does not say. The very small model says yes to the
no-show question and yes to the dog, although its prompt held the right chunk for the first and no dog rule for
the second. Retrieval was the same; the model did not follow the excerpts.

In [9]:
from langchain_core.messages import AIMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda


def replay(prompt_value):
    messages = [{"role": role[m.type], "content": m.content} for m in prompt_value.to_messages()]
    answer, note = ask_model(messages)
    return AIMessage(content=answer or f"[{note}]")


llm = RunnableLambda(replay)
chain = up_to_prompt | llm | StrOutputParser()
print("Small hosted model (recorded, through the chain):")
for qid, text in QUESTIONS.items():
    print(f"  {qid}: {chain.invoke(text)}")

# From the lesson's run (lc_pipeline.out.txt): the same chain with llm = ChatOllama(model="gemma3:270m", temperature=0)
TINY_RUN = {
    "q-guests": "Yes, your two clients can come in on Friday afternoon to meet you.",
    "q-noshow": "Yes, you will get the hours back.",
    "q-parcel": "The parcel was delivered to the front desk.",
    "q-dog": "Yes, you can bring your dog to work with you.",
}
print("\nVery small local model (from the lesson's run, not run here):")
for qid, text in TINY_RUN.items():
    print(f"  {qid}: {text}")

Small hosted model (recorded, through the chain):
  q-guests: Yes, your two clients can come in on Friday afternoon to meet you. They must be registered in the app before they arrive, and they need to leave by 6 PM.
  q-noshow: The handbook states that cancellations must be made at least 2 hours before the booking starts to get the hours back. Later cancellations and no-shows still count against your hours. The handbook does not specify exceptions for missed bookings.
  q-parcel: Yes, your parcel should still be at the front desk. The front desk holds mail for 30 days, and your parcel arrived three weeks ago.
  q-dog: The handbook does not say. Please ask the front desk.

Very small local model (from the lesson's run, not run here):
  q-guests: Yes, your two clients can come in on Friday afternoon to meet you.
  q-noshow: Yes, you will get the hours back.
  q-parcel: The parcel was delivered to the front desk.
  q-dog: Yes, you can bring your dog to work with you.


## 10. Make the vectors yourself

So far the vectors came from the lesson's run. Ollama's all-minilm is the model `all-MiniLM-L6-v2`, so you can
make the vectors here with sentence-transformers (the download is about 90 MB). `revision` pins the exact model
files. `normalize_embeddings=True` scales every vector to length 1.

**Look for:** the new vectors differ from the stored ones only in the fourth decimal place or later, and the top 2
chunks for every question are the same as in step 4. The scores may differ by a few ten-thousandths.

In [10]:
import os, warnings
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
warnings.filterwarnings("ignore")
import torch
import transformers
from huggingface_hub.utils import logging as hub_logging
from sentence_transformers import SentenceTransformer

transformers.logging.set_verbosity_error()
hub_logging.set_verbosity_error()
torch.manual_seed(0)  # the model only reads, nothing is random; seeded anyway

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2",
                            revision="1110a243fdf4706b3f48f1d95db1a4f5529b4d41", device="cpu")
def embed(texts):
    return model.encode(texts, normalize_embeddings=True).astype(np.float64)

my_handbook_vecs = embed(chunk_texts)
print("largest difference from the stored chunk vectors:", round(float(np.abs(my_handbook_vecs - handbook_vecs).max()), 4))
for qid, text in QUESTIONS.items():
    s = my_handbook_vecs @ embed([text])[0]
    print(qid, [(chunk_ids[i], round(float(s[i]), 4)) for i in np.argsort(-s)[:2]])

largest difference from the stored chunk vectors: 0.0003
q-guests [('guests-daily', 0.4327), ('rooms-booking', 0.3983)]
q-noshow [('rooms-cancel', 0.6602), ('rooms-booking', 0.5231)]
q-parcel [('mail-holding', 0.5445), ('mail-arrival', 0.5315)]
q-dog [('mail-arrival', 0.1677), ('mail-forward', 0.1673)]


## Now change it

Three small experiments. Each cell has one line marked `# <- change this`. Edit it and run the cell again.

### Experiment 1: a bigger k

Set `k` to 3 and run. The third chunk for the parcel question is mail-forward (0.3181). It is not needed, and it
makes the prompt longer. `ask_model` then finds no recording: the messages are no longer the exact ones that were
sent. Any change to the prompt is a new call to the model.

In [11]:
k = 2  # <- change this
messages = ask("q-parcel", k)
print("chunks in the prompt:", messages[1]["content"].count("<chunk "))
print("prompt length:", len(messages[1]["content"]), "characters")
answer, note = ask_model(messages)
print("answer:", answer or f"[{note}]")

chunks in the prompt: 2
prompt length: 481 characters
answer: Yes, your parcel should still be at the front desk. The front desk holds mail for 30 days, and your parcel arrived three weeks ago.


### Experiment 2: your own question

Write any member question and run. The cell embeds it with the model from step 10 and shows the top 2 chunks and
their scores. A question about the Wi-Fi should put the wifi chunk first with a clear lead. A question the
handbook does not cover should give low scores, and 2 chunks still come back.

In [12]:
my_question = "When does the Wi-Fi password change?"  # <- change this
s = my_handbook_vecs @ embed([my_question])[0]
for rank, i in enumerate(np.argsort(-s)[:3], 1):
    print(f"{rank}. {chunk_ids[i]:<14} {s[i]:.4f}" + ("   (not sent with k = 2)" if rank > 2 else ""))

1. wifi           0.8540
2. rooms-booking  0.2908
3. access         0.2747   (not sent with k = 2)


### Experiment 3: add the missing rule

The handbook says nothing about pets, so the dog question got two mail chunks. Add a chunk with a pet rule and
run. The new chunk should now come first for the dog question, with a much higher score than 0.17. Retrieval can
only find what the handbook holds. Change the text to a rule about something else, and the dog question goes
back to low scores.

In [13]:
new_chunk = ("pets", "Pets are not allowed in the building. Guide dogs and assistance dogs are welcome.")  # <- change this
ids2 = chunk_ids + [new_chunk[0]]
vecs2 = np.vstack([my_handbook_vecs, embed([new_chunk[1]])])
s = vecs2 @ embed([QUESTIONS["q-dog"]])[0]
for i in np.argsort(-s)[:2]:
    print(f"retrieved: {ids2[i]:<14} score {s[i]:.4f}")

retrieved: pets           score 0.5245
retrieved: mail-arrival   score 0.1677


## Optional: if you have your own API key

You never need this. If you have an OpenRouter key, store it in Colab's Secrets panel (the key icon on the left)
as `OPENROUTER_API_KEY` and run the cell: it sends the k = 3 parcel prompt from experiment 1 to the same small
model, so you can see an answer that was not recorded. A few cents at most. Without a key the cell does nothing.

In [14]:
import os, urllib.request
key = os.environ.get("OPENROUTER_API_KEY")
try:
    from google.colab import userdata
    key = key or userdata.get("OPENROUTER_API_KEY")
except Exception:
    pass
if not key:
    print("No key set: skipped. The notebook does not need it.")
else:
    body = {"model": "google/gemini-2.5-flash-lite", "messages": ask("q-parcel", 3), "max_tokens": 200, "temperature": 0}
    req = urllib.request.Request("https://openrouter.ai/api/v1/chat/completions", data=json.dumps(body).encode(),
                                 headers={"Content-Type": "application/json", "Authorization": "Bearer " + key})
    with urllib.request.urlopen(req, timeout=60) as r:
        print(json.loads(r.read())["choices"][0]["message"]["content"])

No key set: skipped. The notebook does not need it.


## What you should see

- The vectors cell prints 10 chunk vectors, 384 numbers each, and the first vector's length is 1.0.
- For the parcel question the top two chunks are mail-holding and mail-arrival, with scores close to each other
  (0.5442 and 0.5318 with all-minilm).
- For the dog question the best score is far lower (0.1675) than for the other three, and two chunks still come
  back.
- The LangChain cell prints "identical" for all four questions: it built the same messages as the from-scratch
  code.
- The very small local model's answers (from the lesson's run) say dogs are welcome, while the recorded hosted model
  says the handbook does not say: that is the model, not retrieval.